In [3]:
import json
import sys
import os
import time
from tqdm import tqdm
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
import importlib
import csv


Since we calculated the voting scores in batches, we'll need to count the individual principle performance across each batch file and aggregate at the end. 

In [18]:
# We can turn the prompt dataset into a dictionary for easier access

prompt_dataset_filepath = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Princ_from_reasons\data_subset.json'

with open(prompt_dataset_filepath, 'r') as f:
    prompt_dataset = json.load(f)



prompt_dict = {}
for i, prompt in enumerate(prompt_dataset):
    prompt_dict[prompt['prompt']] = i
    i += 1

print(prompt_dataset[0]['all_preferences_unprocessed'])


[{'strength': -1, 'justification': '@Response 1 is slightly better than @Response 2. Both both responses are correct, well written, and do a good job explaining a very complicated topic.\n\n-Verbosity - @Response 2 presents the same information in two ways, which is unnecessary and makes the response overly long.'}, {'strength': -1, 'justification': '@Response 1 edges out slightly over @Response 2 because it offers a broader view of potential applications and slightly more context on why nested genetic algorithms are beneficial, especially in terms of dealing with complex problems and the manual difficulty of determining parameters. This adds a bit more depth and utility to the explanation, making it slightly more comprehensive.'}, {'strength': -1, 'justification': '@Response 1 is better than @Response 2 because it is concise.\n\n\n\nVerbosity: @Response 2 is more verbose; it provides more details about nested genetic algorithms. @Response 1, on the other hand, provides a concise and s

In [19]:
principles_filepath = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Principle Generations\11_26_updated_summarizing\11_26_summarized_principles.json'

with open(principles_filepath, 'r') as f:
    principles = json.load(f)

print(len(principles))



506


In [42]:
def process_score_batch(filepath, score_list, num_irrelevant_list):
    with open(filepath, 'r', encoding='utf-8') as f:
        score_batch = json.load(f)

    temp_scores = [] * len(principles)
    for response in score_batch:
        LLM_annotation = response['annotation']
        prompt_id = prompt_dict[response['prompt']]
        prompt_data = prompt_dataset[prompt_id]
        principle_id = response['principle_index']
        annotator_scores = prompt_data['all_preferences_unprocessed']

        if LLM_annotation:
            for human_annotation in annotator_scores:
                if human_annotation['strength'] < 0 and LLM_annotation == 1:
                    score_list[principle_id] += 1
                elif human_annotation['strength'] > 0 and LLM_annotation == 2:
                    score_list[principle_id] += 1
        else:
            num_irrelevant_list[principle_id] += 1



In [61]:
score_directory = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Principle Scoring\batch_scores'
aggregate_scores = [0] * len(principles)
num_irrelevant = [0] * len(principles)

# Iterate through score files and add them to aggregate scores
for score_file in os.listdir(score_directory):
    full_path = os.path.join(score_directory, score_file)
    process_score_batch(full_path, aggregate_scores, num_irrelevant)

# Find the total number of annotations (sum of all annotators within each prompt)
total_annotations = 0
for prompt in prompt_dataset:
    total_annotations += len(prompt['all_preferences_unprocessed'])

print(f'Total annotations: {total_annotations}')
print(f'Sum of scores: {(aggregate_scores[0])}')

# Calculate the final scores
final_scores = [0] * len(principles)
for i in range(len(aggregate_scores)):
    final_scores[i] = (i, (aggregate_scores[i] / (((total_annotations) - num_irrelevant[i]) + 9)))

final_scores.sort(key=lambda x: x[1], reverse=True)

print(final_scores)

for score in final_scores[:15]:
    print(f'Principle: {principles[score[0]]['summarized_principle']} \nScore: {score[1]} \n')

Total annotations: 2359
Sum of scores: 1378
[(222, 0.6307562315166878), (1, 0.6261089987325729), (430, 0.6243117323168149), (202, 0.6236240474174428), (52, 0.6228813559322034), (441, 0.6221939855993224), (61, 0.6218842416561048), (190, 0.6208791208791209), (26, 0.6194503171247357), (189, 0.6171742808798646), (97, 0.6163976210705183), (231, 0.6162299239222316), (47, 0.6156448202959831), (182, 0.6153846153846154), (206, 0.6152871621621622), (361, 0.614702154626109), (89, 0.6134347275031685), (453, 0.6131078224101479), (219, 0.6130825138948268), (462, 0.6130397967823878), (121, 0.6130122517955218), (503, 0.6126164267569856), (104, 0.6113703860840051), (488, 0.6113223489649345), (11, 0.6111584327086882), (280, 0.6110641891891891), (378, 0.6106644096487516), (289, 0.6098388464800678), (79, 0.6094674556213018), (338, 0.6094674556213018), (334, 0.609375), (96, 0.6088060965283658), (180, 0.6082910321489001), (199, 0.6077933079203727), (81, 0.6077020736352094), (451, 0.6075200675961132), (370, 